# Model Benchmark

In [ ]:
import os, pickle, sys, warnings
from pathlib import Path
import sys
os.environ['PYTHONWARNINGS'] = 'ignore:pkg_resources is deprecated as an API:UserWarning'
warnings.filterwarnings('ignore', message=r'pkg_resources is deprecated as an API.*', category=UserWarning)
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc
from tqdm.auto import tqdm
from ReCAST import ReCAST
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
import seaborn as sns

In [ ]:
project_path = Path('/home/boscoll/Projects/cdk_predict')
train_test_split = project_path / 'data/development_cohort/train_test_split/genomic_train_test_split_26Feb_2026'
clinical_path = project_path / 'data/development_cohort/clinical/extended_clinical_features_cdk_1L_26Feb.csv'


with open(train_test_split, 'rb') as f:
    X_genomic, X_holdout_locked, y_development, y_holdout_locked = pickle.load(f)
y_development = y_development.loc[X_genomic.index, ['Time', 'Event']].copy()

clinical = pd.read_csv(clinical_path, index_col=0)

#from the clinical only dataframe, remove the 'met_sample' variable
X_clinical = clinical.loc[X_genomic.index].copy()
X_clinical.drop(columns='met_sample', inplace=True)

X_feature_sets = {
    'clinical': X_clinical,
    'genomic': X_genomic,
    'clinicogenomic': pd.concat([clinical.loc[X_genomic.index], X_genomic], axis=1),
}

In [ ]:
#HELPER FUNCTIONS 

#normalize to 1st and 99th percentile risk-scores using the training folds distribution. ReCAST already performs this normalization internally
def normalize_from_outer_train(train_risk, outer_risk):
    lower, upper = np.quantile(np.asarray(train_risk), [0.01, 0.99])
    if upper <= lower:
        return np.zeros(len(outer_risk))
    return np.clip((np.clip(outer_risk, lower, upper) - lower) / (upper - lower) * 100, 0, 100)

#y format for sksurv library
def y_forSurv(data):
    '''
    Transform the y variable into the format required by sksurv
    data ---> DataFrame containing 'Event' and 'Time' columns and IDs as index
    return ---> y variable in the format required by sksurv
    '''
    from sksurv.util import Surv
    return Surv.from_dataframe("Event", "Time", data.reset_index(drop=True))

#cindex for xgboost
def xgb_cindex(y_signed, risk):
    y_signed = np.asarray(y_signed)
    return concordance_index_censored(y_signed > 0, np.abs(y_signed), risk)[0]
xgb_scorer = make_scorer(xgb_cindex)

In [ ]:
#GRID FOR HYPERP. TUNING
elastic_net_grid = {
    'l1_ratio': np.round(np.arange(0.1, 1.0, 0.1), 1),
    'alphas': [[0.001], [0.005], [0.01], [0.05]],
}
lasso_grid = {'alphas': [[0.001], [0.005], [0.01], [0.05]]}
rsf_grid = {
    'max_depth': [3, 6, 9], 'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [5, 10, 20], 'max_features': ['sqrt', 0.5],
    'n_estimators': [100, 500],
}
xgb_grid = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2], 'max_depth': [3, 6, 9],
    'subsample': [0.7, 1.0], 'colsample_bytree': [0.6, 1.0],
    'reg_alpha': [0.0, 0.1], 'reg_lambda': [0.1, 1.0],
    'min_child_weight': [1, 5], 'max_delta_step': [1.0, 5.0],
    'n_estimators': [100, 500, 1000]
}

In [ ]:
#model names +features sets
expected_columns = [
    f'{feature_set}__{model}'
    for feature_set in X_feature_sets
    for model in ['ReCAST', 'OncoCast', 'Elastic Net Cox', 'Lasso Cox', 'Random survival forest', 'XGBoost Cox']
]

#5-fold splitter stratified for PFS Event
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=94)
outer_splits = list(
    outer_cv.split(
        np.zeros(len(y_development)),
        y_development['Event'].astype(int), 
    )
)

oof_predictions = pd.DataFrame(index=y_development.index)

for feature_set, X_data in tqdm(
    X_feature_sets.items(),
    total=len(X_feature_sets),
    desc='Feature blocks',
):
    oof = {model: np.full(len(X_data), np.nan) for model in ['ReCAST', 'OncoCast', 'Elastic Net Cox', 'Lasso Cox', 'Random survival forest', 'XGBoost Cox']}

    for outer_fold, (train_pos, test_pos) in enumerate(outer_splits):
        X_outer_train = X_data.iloc[train_pos]
        X_outer_test = X_data.iloc[test_pos]
        y_outer_train = y_development.iloc[train_pos]

        recast = ReCAST(
            n_models=100,
            l1=1,
            n_folds=3,
            bootstrap=True,
            random_state=94 + outer_fold,
            adaptive_lasso=True,
            normalization=(0.01, 0.99),
            metric='auc',
            auc_time_range=(3, 36),
            subagging=1,
            n_jobs=-2,
            verbose=False,
        )
        recast.fit(X_outer_train, y_outer_train)
        oof['ReCAST'][test_pos] = recast.predict(
            X_outer_test.copy(),
            verbose=False,
        ).to_numpy()

        #by setting ReCAST to turn off adaptive lasso and bootstrap resampling, reproduce OncoCast with L1 penalty
        oncocast = ReCAST( 
            n_models=100,
            l1=1,
            n_folds=3,
            bootstrap=False,
            random_state=94 + outer_fold,
            adaptive_lasso=False,
            metric='cindex',
            subagging=1,
            n_jobs=-2,
            verbose=False,
        )
        oncocast.fit(X_outer_train, y_outer_train)
        oof['OncoCast'][test_pos] = oncocast.predict(
            X_outer_test.copy(),
            verbose=False,
        ).to_numpy()

        #for standard ML models, nested cross validation using 3 inner folds for hyperp. tuning, still stratified for PFS Event
        inner_cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=94 + outer_fold,
        )
        inner_splits = list(
            inner_cv.split(
                X_outer_train,
                y_outer_train['Event'].astype(int),
            )
        )
        y_outer_surv = y_forSurv(y_outer_train)

        elastic_net = GridSearchCV(
            CoxnetSurvivalAnalysis(l1_ratio=0.5, max_iter=100_000),
            elastic_net_grid,
            cv=inner_splits,
            n_jobs=-1,
            error_score=np.nan,
        ).fit(X_outer_train, y_outer_surv)
        oof['Elastic Net Cox'][test_pos] = normalize_from_outer_train(
            elastic_net.best_estimator_.predict(X_outer_train),
            elastic_net.best_estimator_.predict(X_outer_test),
        )

        lasso = GridSearchCV(
            CoxnetSurvivalAnalysis(l1_ratio=1.0, max_iter=100_000),
            lasso_grid,
            cv=inner_splits,
            n_jobs=-1,
            error_score=np.nan,
        ).fit(X_outer_train, y_outer_surv)
        oof['Lasso Cox'][test_pos] = normalize_from_outer_train(
            lasso.best_estimator_.predict(X_outer_train),
            lasso.best_estimator_.predict(X_outer_test),
        )

        rsf = RandomizedSearchCV(
            RandomSurvivalForest(
                # n_estimators=100,
                n_jobs=1,
                random_state=94 + outer_fold,
            ),
            rsf_grid,
            n_iter=10,
            cv=inner_splits,
            n_jobs=-1,
            random_state=94 + outer_fold,
            error_score=np.nan,
        ).fit(X_outer_train, y_outer_surv)
        oof['Random survival forest'][test_pos] = normalize_from_outer_train(
            rsf.best_estimator_.predict(X_outer_train),
            rsf.best_estimator_.predict(X_outer_test),
        )

        y_outer_xgb = np.where(
            y_outer_train['Event'].astype(bool),
            y_outer_train['Time'],
            -y_outer_train['Time'],
        )
        xgb_model = RandomizedSearchCV(
            xgb.XGBRegressor(
                objective='survival:cox',
                # n_estimators=100,
                tree_method='hist',
                n_jobs=1,
                random_state=94 + outer_fold,
                verbosity=0,
            ),
            xgb_grid,
            n_iter=10,
            scoring=xgb_scorer,
            cv=inner_splits,
            n_jobs=-1,
            random_state=94 + outer_fold,
            error_score=np.nan,
        ).fit(X_outer_train, y_outer_xgb)
        oof['XGBoost Cox'][test_pos] = normalize_from_outer_train(
            xgb_model.best_estimator_.predict(X_outer_train),
            xgb_model.best_estimator_.predict(X_outer_test),
        )

    for model, risk in oof.items():
        oof_predictions[f'{feature_set}__{model}'] = risk

Feature blocks: 100%|██████████| 3/3 [10:06<00:00, 202.27s/it]


In [12]:
# oof_predictions.to_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/oof_predictions_MLmodels.csv')
oof_predictions = pd.read_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/oof_predictions_MLmodels.csv')

## transformers results import

In [ ]:
tabfn = pd.read_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabpfn_risks.csv')
clinical_transformer = pd.read_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/clinical_transformer_risks.csv')
tabfm = pd.read_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabfm_risks.csv')
tabfm.columns = tabfm.columns.str.replace('_v1_0_0', '')

#merge all models
oof_predictions = oof_predictions.merge(tabfn, on='record_id').merge(clinical_transformer, on ='record_id').merge(tabfm, on='record_id')

## metrics

In [15]:
tabpfn_names = tabfn.iloc[:, 1:].columns.str.split('__').str[1].unique().to_list()
clinical_transformer_names = clinical_transformer.iloc[:, 1:].columns.str.split('__').str[1].unique().to_list()
tabfm_names = tabfm.iloc[:, 1:].columns.str.split('__').str[1].unique().to_list()

In [ ]:
MODEL_NAMES = ['ReCAST', 'OncoCast', 'Elastic Net Cox', 'Lasso Cox', 'Random survival forest', 'XGBoost Cox']
MODEL_NAMES = MODEL_NAMES + tabpfn_names + clinical_transformer_names + tabfm_names
expected_columns = [
    f'{feature_set}__{model}'
    for feature_set in X_feature_sets
    for model in MODEL_NAMES
]

**cindex and cumulative auc**

In [19]:
#clinical transformer has only clinicogenomic, remove clinical and genomic feature sets from expected columns
expected_columns = [col for col in expected_columns if not col.startswith('clinical__clinical') and not col.startswith('genomic__clinical')]

In [ ]:
# point estimates for OOF predictions.
y_reference = y_forSurv(y_development)
point_rows = []

for column in expected_columns:
    feature_set, model = column.split('__', 1)
    risk = oof_predictions[column].to_numpy()

    cindex = concordance_index_censored(
        y_development['Event'].astype(bool),
        y_development['Time'],
        risk,
    )[0]
    _, mean_auc = cumulative_dynamic_auc(
        y_reference,
        y_reference,
        risk,
        times=np.arange(6, 61, 6),
    )
    point_rows.append({
        'feature_set': feature_set,
        'model': model,
        'cindex': float(cindex),
        'mean_auc_6_60': float(mean_auc),
    })

point_results = pd.DataFrame(point_rows)

#bootstrap n=1000 times 
res = []
for seed in tqdm( np.arange(1, 1001), desc='Paired OOF bootstraps'):
    rng = np.random.default_rng(seed)
    positions = rng.choice(
        len(y_development),
        size=len(y_development),
        replace=True,
    )
    y_boot = y_development.iloc[positions].reset_index(drop=True)

    for column in expected_columns:
        feature_set, model = column.split('__', 1)
        risk_boot = oof_predictions[column].to_numpy()[positions]

        cindex = concordance_index_censored(
            y_boot['Event'].astype(bool),
            y_boot['Time'],
            risk_boot,
        )[0]
        _, mean_auc = cumulative_dynamic_auc(
            y_reference,
            y_forSurv(y_boot),
            risk_boot,
            times=np.arange(6, 61, 6),
        )
        res.append({
            'seed': int(seed),
            'feature_set': feature_set,
            'model': model,
            'cindex': float(cindex),
            'mean_auc_6_60': float(mean_auc),
        })

bootstrap_results = pd.DataFrame(res)

Paired OOF bootstraps: 100%|██████████| 1000/1000 [08:36<00:00,  1.94it/s]


**contrast score using elasticnet as reference**

In [ ]:
ref_model = 'Elastic Net Cox'

intervals = bootstrap_results.groupby(['feature_set', 'model']).agg(
    cindex_low=('cindex', lambda x: x.quantile(0.025)),
    cindex_high=('cindex', lambda x: x.quantile(0.975)),
    mean_auc_low=('mean_auc_6_60', lambda x: x.quantile(0.025)),
    mean_auc_high=('mean_auc_6_60', lambda x: x.quantile(0.975)),
    valid_bootstraps=('seed', 'nunique'),
).reset_index()
summary = point_results.merge(intervals, on=['feature_set', 'model'], validate='one_to_one')

paired_rows = []
for metric in ['cindex', 'mean_auc_6_60']:
    wide = bootstrap_results.pivot(
        index=['seed', 'feature_set'], columns='model', values=metric
    )
    for model in MODEL_NAMES:
        if model == ref_model:
            continue
        delta = wide[model] - wide[ref_model]
        for feature_set in X_feature_sets:
            values = delta.xs(feature_set, level='feature_set').dropna()
            paired_rows.append({
                'feature_set': feature_set, 'model': model, 'reference': ref_model,
                'metric': metric, 'delta_mean': values.mean(),
                'delta_low': values.quantile(0.025), 'delta_high': values.quantile(0.975),
                'probability_positive': (values > 0).mean(),
                'valid_bootstraps': values.size,
            })
paired_deltas = pd.DataFrame(paired_rows)


***Tables for standard ML models benchmark***

In [ ]:
#exclude tabpfn_names + clinical_transformer_names + tabfn_names
feature_order = ['clinicogenomic', 'clinical', 'genomic']
feature_labels = {
    feature_set: f'{feature_set.capitalize()} — all {X_feature_sets[feature_set].shape[1]} features'
    for feature_set in feature_order
}

tables = {}
for sheet, metric, low, high, label in [
    ('C-index', 'cindex', 'cindex_low', 'cindex_high', 'C-index'),
    ('Mean AUC 6-60', 'mean_auc_6_60', 'mean_auc_low', 'mean_auc_high', 'Mean AUC[6–60]'),
]:
    deltas = paired_deltas.query('metric == @metric and model not in @tabpfn_names and model not in @clinical_transformer_names and model not in @tabfn_names')[
        ['feature_set', 'model', 'delta_mean', 'probability_positive']
    ]

    table = summary.query('model not in @tabpfn_names and model not in @clinical_transformer_names and model not in @tabfn_names').merge(
        deltas,
        on=['feature_set', 'model'],
        how='left',
        validate='one_to_one',
    )

    table['feature_set'] = pd.Categorical(
        table['feature_set'],
        categories=feature_order,
        ordered=True,
    )

    table = table.sort_values(
        ['feature_set', metric],
        ascending=[True, False],
        na_position='last',
    )

    table['Feature block'] = table['feature_set'].map(feature_labels)
    table['95% CI'] = table.apply(
        lambda row: f"{row[low]:.3f}–{row[high]:.3f}",
        axis=1,
    )

    table = table[[
        'Feature block',
        'model',
        metric,
        '95% CI',
        'delta_mean',
        'probability_positive',
    ]].rename(columns={
        'model': 'Arm',
        metric: label,
        'delta_mean': f'Δ vs {ref_model}',
        'probability_positive': 'P(Δ > 0)',
    })

    numeric_columns = [
        label,
        f'Δ vs {ref_model}',
        'P(Δ > 0)',
    ]
    table[numeric_columns] = table[numeric_columns].round(3)

    tables[sheet] = table.reset_index(drop=True)

# tables['Mean AUC 6-60'].to_excel('/home/boscoll/Projects/cdk_predict/results/tables/auc_ML_models.xlsx')
# tables['C-index'].to_excel('/home/boscoll/Projects/cdk_predict/results/tables/cindex_ML_models.xlsx')

**contrast score using ReCAST as reference and using foundation models**

In [ ]:
MODEL_NAMES_transformers = tabpfn_names + clinical_transformer_names + tabfm_names + ['ReCAST']
ref_transformers = 'ReCAST'

In [ ]:
intervals = bootstrap_results.groupby(['feature_set', 'model']).agg(
    cindex_low=('cindex', lambda x: x.quantile(0.025)),
    cindex_high=('cindex', lambda x: x.quantile(0.975)),
    mean_auc_low=('mean_auc_6_60', lambda x: x.quantile(0.025)),
    mean_auc_high=('mean_auc_6_60', lambda x: x.quantile(0.975)),
    valid_bootstraps=('seed', 'nunique'),
).reset_index()
summary = point_results.merge(intervals, on=['feature_set', 'model'], validate='one_to_one')

paired_rows_transformers = []
for metric in ['cindex', 'mean_auc_6_60']:
    wide = bootstrap_results.pivot(
        index=['seed', 'feature_set'], columns='model', values=metric
    )
    for model in MODEL_NAMES_transformers:
        if model == ref_transformers:
            continue
        delta = wide[model] - wide[ref_transformers]
        for feature_set in X_feature_sets:
            values = delta.xs(feature_set, level='feature_set').dropna()
            paired_rows_transformers.append({
                'feature_set': feature_set, 'model': model, 'reference': ref_transformers,
                'metric': metric, 'delta_mean': values.mean(),
                'delta_low': values.quantile(0.025), 'delta_high': values.quantile(0.975),
                'probability_positive': (values > 0).mean(),
                'valid_bootstraps': values.size,
            })
paired_deltas_transformers = pd.DataFrame(paired_rows_transformers)


**extended_data_fig_03**

In [ ]:
_, (ax1, ax2, ax3) = plt.subplots(figsize=(14, 2.2), nrows = 1, ncols = 3, sharex = True)

models_plot = ['ReCAST'] + tabpfn_names + clinical_transformer_names + tabfm_names
models_plot = models_plot[::-1]
models_plot_no_clinical_transformer = ['ReCAST'] + tabpfn_names + tabfm_names
models_plot_no_clinical_transformer = models_plot_no_clinical_transformer[::-1]

#clinical bloc
df_plot = summary.query('feature_set == "clinical" and model in @models_plot')
#sort following models_plot order
df_plot = df_plot.set_index('model').loc[models_plot_no_clinical_transformer].reset_index()
df_plot.dropna(inplace = True)
ax1.set_xlim(0.58, 0.78)
for _, r in df_plot.iterrows():
    ax1.plot([r['mean_auc_low'], r['mean_auc_high']], [r['model'], r['model']],
            color='#ef3e2d', lw=1, solid_capstyle='round', zorder=2)
    ax1.scatter(r['mean_auc_6_60'], r['model'],
                   s=50, marker='o',
                   color= '#ef3e2d',
                   edgecolor='white', linewidth=0.4,
                   zorder=3)
ax1.grid(True, linestyle='--', linewidth=0.5, alpha=0.7, axis='x')
ax1.set_xticks([0.60, 0.65, 0.70, 0.75])
ax1.set_title('Clinical block', fontsize = 10)
for _, r in df_plot.iterrows():
    ax1.text(0.78, r['model'],
             f"{r['mean_auc_6_60']:.3f} ({r['mean_auc_low']:.3f}-{r['mean_auc_high']:.3f})",
             va='center')
ax1.set_xlabel('AUC[t] (95%CI)')


#genomic block
df_plot = summary.query('feature_set == "genomic" and model in @models_plot')
df_plot = df_plot.set_index('model').loc[models_plot_no_clinical_transformer].reset_index()
df_plot.dropna(inplace = True)
ax2.set_xlim(0.58, 0.78)
for _, r in df_plot.iterrows():
    ax2.plot([r['mean_auc_low'], r['mean_auc_high']], [r['model'], r['model']],
            color='#f69274', lw=1, solid_capstyle='round', zorder=2)
    ax2.scatter(r['mean_auc_6_60'], r['model'],
                   s=50, marker='o',
                   color= '#f69274',
                   edgecolor='white', linewidth=0.4,
                   zorder=3)
ax2.grid(True, linestyle='--', linewidth=0.5, alpha=0.7, axis='x')
ax2.set_xticks([0.58, 0.60, 0.65, 0.70, 0.75])
ax2.set_title('Genomic block', fontsize = 10)
for _, r in df_plot.iterrows():
    ax2.text(0.78, r['model'],
             f"{r['mean_auc_6_60']:.3f} ({r['mean_auc_low']:.3f}-{r['mean_auc_high']:.3f})",
             va='center')
ax2.set_xlabel('AUC[t] (95%CI)')

#clinicogenomic 
df_plot = summary.query('feature_set == "clinicogenomic" and model in @models_plot')
df_plot = df_plot.set_index('model').loc[models_plot].reset_index()
ax3.set_xlim(0.58, 0.78)
#ci bar
for _, r in df_plot.iterrows():
    # CI bar
    ax3.plot([r['mean_auc_low'], r['mean_auc_high']], [r['model'], r['model']],
            color='#9a1b1f', lw=1, solid_capstyle='round', zorder=2)
    ax3.scatter(r['mean_auc_6_60'], r['model'],
                   s=50, marker='o',
                   color= '#9a1b1f',
                   edgecolor='white', linewidth=0.4,
                   zorder=3)
ax3.grid(True, linestyle='--', linewidth=0.5, alpha=0.7, axis='x')
#x ticks at 0.60 and 0.65, and 0.70 
ax3.set_xticks([0.60, 0.65, 0.70, 0.75])
ax3.set_title('Clinicogenomic block', fontsize = 10)
#on the right of the plot, put the c index, and the 95%CI in (x-x)
for _, r in df_plot.iterrows():
    ax3.text(0.78, r['model'],
             f"{r['mean_auc_6_60']:.3f} ({r['mean_auc_low']:.3f}-{r['mean_auc_high']:.3f})",
             va='center')
ax3.set_xlabel('AUC[t] (95%CI)', fontsize = 10)

sns.despine()
plt.tight_layout()

# plt.savefig('/home/boscoll/Projects/cdk_predict/results/images/paper_figures/models_benchmarks/auc_models.pdf')
plt.show()

**Fig. 2D**

In [ ]:
#paired deltas with auc
_, (ax1) = plt.subplots(ncols = 1, nrows= 1, figsize=(4.5, 2), sharex= True)

max = paired_deltas_transformers.query('metric == "mean_auc_6_60"').delta_high.max()

#clinicogenomic block
df_plot = paired_deltas_transformers.query('metric == "mean_auc_6_60" and feature_set == "clinicogenomic"')
df_plot = df_plot.dropna()
df_plot = df_plot.sort_values('delta_mean')
ax1.set_xlim(-0.04, 0.045)
for _, r in df_plot.iterrows():
    ax1.plot([r['delta_low'], r['delta_high']], [r['model'], r['model']],
            color='#9a1b1f', lw=1, solid_capstyle='round', zorder=2)
    ax1.scatter(r['delta_mean'], r['model'],
                   s=50, marker='o',
                   color= '#9a1b1f',
                   edgecolor='white', linewidth=0.4,
                   zorder=3)
ax1.axvline(0, color='black', linestyle='-', lw=1)
ax1.set_xticks([-0.02, 0, 0.02])
for _, r in df_plot.iterrows():
    ax1.text(max, r['model'],
             f"P(Δ > 0) {r['probability_positive']:.2f}",
             va='center')
ax1.set_title('', fontsize = 10)
ax1.set_xlabel('(t)AUC Δ vs ReCAST')
sns.despine()

plt.tight_layout()
# plt.savefig('/home/boscoll/Projects/cdk_predict/results/images/paper_figures/models_benchmarks/delta_models.pdf')
plt.show()

# Cross-fitted feature-deletion analysis

In [ ]:
reliance_mutations = [
    'BRCA1_tsg', 'BRCA2_tsg','RB1_tsg', 'PALB2_tsg',
]

In [ ]:
project_path = Path('/home/boscoll/Projects/cdk_predict')
train_test_split = project_path / 'data/development_cohort/train_test_split/genomic_train_test_split_26Feb_2026'
clinical_path = project_path / 'data/development_cohort/clinical/extended_clinical_features_cdk_1L_26Feb.csv'

with open(train_test_split, 'rb') as f:
    X_genomic, X_holdout_locked, y_development, y_holdout_locked = pickle.load(f)
X_genomic_all = pd.concat([X_genomic, X_holdout_locked], axis=0)
y_reliance = pd.concat([y_development, y_holdout_locked[['Time', 'Event']]], axis=0)

clinical = pd.read_csv(clinical_path, index_col=0)

X_reliance = pd.concat([clinical.loc[X_genomic_all.index], X_genomic_all], axis=1).astype(float)

In [ ]:
reliance_rows = []

for gene_nr, gene in enumerate(tqdm(reliance_mutations,  desc='Alterations')):
    for repeat in tqdm(range(50), desc=gene,  leave=False):
        outer_cv = StratifiedKFold(
            n_splits=5,shuffle=True, random_state=94 + repeat
        )
        outer_splits = outer_cv.split(
            X_reliance, X_reliance[gene].astype(int)
        )

        for outer_fold, (train_pos, test_pos) in enumerate(outer_splits):
            seed = 94 + gene_nr * 10_000 + repeat * 10 + outer_fold
            X_outer_train = X_reliance.iloc[train_pos]
            X_outer_test = X_reliance.iloc[test_pos]
            y_outer_train = y_reliance.iloc[train_pos]

            # Only true carriers in the outer-test fold are ablated.
            X_carriers = X_outer_test.loc[X_outer_test[gene]== 1].copy()
            X_ablated = X_carriers.copy()
            X_ablated[gene] = 0

            # ReCAST and OncoCast
            recast = ReCAST(
                n_models=100, l1=1, n_folds=3, bootstrap=True,
                random_state=seed, adaptive_lasso=True,
                normalization=(0.01, 0.99), metric='auc',
                auc_time_range=(3, 36), subagging=1, n_jobs=-2,
                verbose=False,
            )
            recast.fit(X_outer_train,y_outer_train)

            oncocast = ReCAST(
                n_models=100, l1=1, n_folds=3, bootstrap=False,
                random_state=seed, adaptive_lasso=False,
                metric='cindex', subagging=1, n_jobs=-2, verbose=False,
            )
            oncocast.fit(X_outer_train, y_outer_train)

            # Inner three-fold CV for comparator models.
            inner_cv = StratifiedKFold(
                n_splits=3, shuffle=True, random_state=seed
            )
            inner_splits = list(inner_cv.split(
                X_outer_train, y_outer_train['Event'].astype(int)
            ))
            y_outer_surv = y_forSurv(y_outer_train)

            elastic_net = GridSearchCV(
                CoxnetSurvivalAnalysis(l1_ratio=0.5, max_iter=100_000),
                elastic_net_grid, cv=inner_splits, n_jobs=-1,
                error_score=np.nan,
            ).fit(X_outer_train, y_outer_surv)

            lasso = GridSearchCV(
                CoxnetSurvivalAnalysis(l1_ratio=1.0, max_iter=100_000),
                lasso_grid, cv=inner_splits, n_jobs=-1,
                error_score=np.nan,
            ).fit(X_outer_train, y_outer_surv)

            rsf = RandomizedSearchCV(
                RandomSurvivalForest(n_jobs=1, random_state=seed),
                rsf_grid, n_iter=10, cv=inner_splits,
                n_jobs=-1, random_state=seed, error_score=np.nan,
            ).fit(X_outer_train, y_outer_surv)

            y_outer_xgb = np.where(
                y_outer_train['Event'].astype(bool),
                y_outer_train['Time'],
                -y_outer_train['Time'],
            )
            xgb_model = RandomizedSearchCV(
                xgb.XGBRegressor(
                    objective='survival:cox', tree_method='hist',
                    n_jobs=1, random_state=seed, verbosity=0,
                ),
                xgb_grid, n_iter=10, scoring=xgb_scorer,
                cv=inner_splits, n_jobs=-1, random_state=seed,
                error_score=np.nan,
            ).fit(X_outer_train, y_outer_xgb)

            models = {
                'ReCAST': recast,
                'OncoCast': oncocast,
                'Elastic Net Cox': elastic_net.best_estimator_,
                'Lasso Cox': lasso.best_estimator_,
                'Random survival forest': rsf.best_estimator_,
                'XGBoost Cox': xgb_model.best_estimator_,
            }

            for model_name, model in models.items(): 
                if model_name in ['ReCAST', 'OncoCast']:
                    risk_observed = model.predict(
                        X_carriers.copy(), verbose=False
                    ).to_numpy()
                    risk_ablated = model.predict(
                        X_ablated.copy(), verbose=False
                    ).to_numpy()
                else:
                    train_risk = model.predict(X_outer_train)
                    risk_observed = normalize_from_outer_train(
                        train_risk, model.predict(X_carriers)
                    )
                    risk_ablated = normalize_from_outer_train(
                        train_risk, model.predict(X_ablated)
                    )

                for record_id, observed, ablated in zip(
                    X_carriers.index, risk_observed, risk_ablated
                ):
                    reliance_rows.append({
                        'record_id': record_id,
                        'gene': gene,
                        'model': model_name,
                        'repeat': repeat,
                        'outer_fold': outer_fold,
                        'risk_observed': observed,
                        'risk_ablated': ablated,
                        'delta': observed - ablated,
                        'absolute_delta': abs(observed - ablated),
                    })

reliance_raw = pd.DataFrame(reliance_rows)
# reliance_raw.to_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/model_reliance/reliance_all_repeats.csv', index=False)

Alterations: 100%|██████████| 5/5 [11:07:11<00:00, 8006.24s/it] 


In [6]:
reliance_raw = pd.read_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/model_reliance/reliance_all_repeats.csv')
reliance_tabfm = pd.read_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabfm_model_reliance/reliance_all_repeats.csv')
reliance_tabpfn = pd.read_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/tabpfn_model_reliance/reliance_all_repeats.csv')

In [7]:
reliance_raw = pd.concat([reliance_raw, reliance_tabfm, reliance_tabpfn])

In [ ]:
# Median over the 50 repeated estimates for each patient
reliance_patients = reliance_raw.groupby(
    ['record_id', 'gene', 'model'], as_index=False
).agg(
    median_delta=('delta', 'median'),
    median_absolute_delta=('absolute_delta', 'median'),
)

reliance_summary = reliance_patients.groupby(
    ['gene', 'model'], as_index=False
).agg(
    n_carriers=('record_id', 'nunique'),
    median_delta=('median_delta', 'median'),
    delta_q25=('median_delta', lambda x: x.quantile(0.25)),
    delta_q75=('median_delta', lambda x: x.quantile(0.75)),
    median_absolute_delta=('median_absolute_delta', 'median'),
)
reliance_patients = reliance_patients.merge(reliance_summary[['gene', 'n_carriers']].drop_duplicates(), on ='gene')



**Fig. 2B**

In [ ]:
_, axes = plt.subplots(nrows = 2, ncols =2, figsize = (6, 4), sharey=True)
#put TabFM, TabPFN v2.5 and TabPFN v3 at the bottom
transformers = ['TabFM', 'TabPFN v2.5', 'TabPFN v3']
order = [i for i in reliance_patients.model.unique() if i not in transformers] 

for nr, i in enumerate([m for m in reliance_mutations]):
    sns.boxplot(data =reliance_patients[reliance_patients['gene']==i], y='model', x='median_delta', ax=axes[nr//2, nr%2], color= 'white', showfliers = False, 
                linewidth=0.5, linecolor = 'black', order=order)
    sns.stripplot(data =reliance_patients[reliance_patients['gene']==i], y='model', x='median_delta', ax=axes[nr//2, nr%2], color = '#8c2d04', size=4, order=order)
    axes[nr//2, nr%2].set_title(f'{i} (n={reliance_patients[reliance_patients["gene"]==i]["n_carriers"].iloc[0]})')
    axes[nr//2, nr%2].axvline(0, color='black', linestyle='-', linewidth = 0.5)
    axes[nr//2, nr%2].set_ylabel('')
    axes[nr//2, nr%2].set_xlabel('Ablated Δ risk-score')
# plt.delaxes(axes[1, 2])
plt.tight_layout()
sns.despine()
# plt.savefig('/home/boscoll/Projects/cdk_predict/results/images/paper_figures/models_benchmarks/model_reliance.pdf')
plt.show()


**Extended Data Fig. 3B**

In [ ]:
_, axes = plt.subplots(nrows = 1, ncols =4, figsize = (10, 2.5), sharey=True)
#put TabFM, TabPFN v2.5 and TabPFN v3 at the bottom
transformers = ['TabFM', 'TabPFN v2.5', 'TabPFN v3']
order = ['ReCAST'] + transformers

for nr, i in enumerate([m for m in reliance_mutations]):
    sns.boxplot(data =reliance_patients[reliance_patients['gene']==i], y='model', x='median_delta', ax=axes[nr], color= 'white', showfliers = False, 
                linewidth=0.5, linecolor = 'black', order=order)
    sns.stripplot(data =reliance_patients[reliance_patients['gene']==i], y='model', x='median_delta', ax=axes[nr], color = '#8c2d04', size=4, order=order)
    axes[nr].set_title(f'{i} (n={reliance_patients[reliance_patients["gene"]==i]["n_carriers"].iloc[0]})')
    axes[nr].axvline(0, color='black', linestyle='-', linewidth = 0.5)
    axes[nr].set_ylabel('')
    axes[nr].set_xlabel('Ablated Δ risk-score')
# plt.delaxes(axes[1, 2])
plt.tight_layout()
sns.despine()
# plt.savefig('/home/boscoll/Projects/cdk_predict/results/images/paper_figures/models_benchmarks/transformers_model_reliance.pdf')
plt.show()
